# Country Rotation — Annual Strategies + Fixes

**Append to existing notebook after Section 7.**

**This addendum adds:**
- Fix for the `BM`/`BE` bug in Section 5 (rerun `ts_sma200_daily` and `ts_sma10mo_faber`)
- Section 9: Annual rolling-12-month strategies (6 new strategies)
  - Annual momentum (xsec, ts, dual)
  - Annual contrarian (3yr, 5yr — flagged underpowered)

**Rolling-12-month implementation:** every month, rebalance 1/12 of capital to a new portfolio held for the next 12 months. Twelve overlapping cohorts at any time. This is the academic-standard annual-frequency test.

## Fix — BM bug in Section 5 (rerun)

In [1]:
# Fixed ts_sma_filter (changed 'BE' -> 'BM')
def ts_sma_filter_fixed(prices, window_days):
    sma = prices[COUNTRIES].rolling(window_days).mean()
    above = (prices[COUNTRIES] > sma).astype(int)
    monthly_signal = above.resample("BM").last()
    return monthly_signal

# 2b: 200-day SMA filter (daily price) — rerun with fix
picks_2b = ts_sma_filter_fixed(px, window_days=200)
avail_200 = available_mask(px[COUNTRIES], n_months_required=10)
picks_2b = picks_2b.where(avail_200, 0)
w_2b = picks_to_weights_with_bil_fill(picks_2b, px.index)
print("=== ts_sma200_daily (rerun with fix) ===")
r = evaluate("ts_sma200_daily", w_2b); print_eval(r, SPY_CAGR); results["ts_sma200_daily"] = r

# 2c: 10-month SMA (Faber) — should already work but rerun to confirm
picks_2c = ts_sma_monthly(px, n_months=10)
avail_10mo = available_mask(px[COUNTRIES], n_months_required=10)
picks_2c = picks_2c.where(avail_10mo, 0)
w_2c = picks_to_weights_with_bil_fill(picks_2c, px.index)
print("\n=== ts_sma10mo_faber (rerun) ===")
r = evaluate("ts_sma10mo_faber", w_2c); print_eval(r, SPY_CAGR); results["ts_sma10mo_faber"] = r

NameError: name 'px' is not defined

## Section 9 — Annual (rolling-12-month) strategies

**Rolling-12-month logic:** every month-end, pick a portfolio that will be held for the next 12 months. At any given moment, the live portfolio is the average of the last 12 monthly cohorts (1/12 weight each).

**Equivalent representation:** for each month-end `t`, build `w_t` (the cohort selected at `t`). The live weight on day `d` is the average of `w_{t-1}, w_{t-2}, ..., w_{t-12}` (last 12 cohorts whose 12-month hold includes day `d`).

In [ ]:
def overlapping_12mo_weights(monthly_cohort_weights, daily_index):
    """
    monthly_cohort_weights: DataFrame indexed by month-end, columns=tickers, values=weight in that cohort.
    Returns daily weights as 1/12 EW of 12 most recent cohorts (whose 12-month hold covers each day).
    """
    cols = list(monthly_cohort_weights.columns)
    w_daily = pd.DataFrame(0.0, index=daily_index, columns=cols)
    cohort_dates = monthly_cohort_weights.index
    
    # For each day, find the 12 most recent cohorts whose hold (12 months from decision) covers that day
    for i, dt in enumerate(daily_index):
        # Cohorts decided in the last 12 months before dt → 12 most recent decisions with decision_date < dt
        relevant = cohort_dates[(cohort_dates < dt) & (cohort_dates >= dt - pd.DateOffset(months=12))]
        if len(relevant) == 0:
            continue
        # Average the cohort weights
        n_cohorts = len(relevant)
        avg_w = monthly_cohort_weights.loc[relevant].mean(axis=0)
        w_daily.loc[dt] = avg_w.values
    return w_daily

print("Annual rebal helper loaded.")

In [ ]:
# Helper: build monthly cohort weights from a picks function
def picks_to_cohort_weights(picks_monthly, universe_cols):
    """Convert binary picks to EW cohort weights."""
    cohort = pd.DataFrame(0.0, index=picks_monthly.index, columns=universe_cols)
    for i, dt in enumerate(picks_monthly.index):
        row = picks_monthly.iloc[i]
        # universe of picks_monthly is COUNTRIES; some cohort universes include BIL
        # we just fill what's there
        n = int(row.sum())
        if n == 0:
            continue
        for c in picks_monthly.columns:
            if row[c] == 1 and c in universe_cols:
                cohort.loc[dt, c] = 1.0 / n
    return cohort

# === 9a: Annual rolling 12-month momentum, top 5 EW ===
picks_9a = xsec_momentum(px, n_months=12, top_k=5)
cohort_9a = picks_to_cohort_weights(picks_9a, COUNTRIES)
w_9a = overlapping_12mo_weights(cohort_9a, px.index)
print("=== annual_xsec_mom_12_top5 ===")
r = evaluate("annual_xsec_mom_12_top5", w_9a); print_eval(r, SPY_CAGR); results["annual_xsec_mom_12_top5"] = r

# === 9b: Annual rolling 12-month momentum, top 3 EW ===
picks_9b = xsec_momentum(px, n_months=12, top_k=3)
cohort_9b = picks_to_cohort_weights(picks_9b, COUNTRIES)
w_9b = overlapping_12mo_weights(cohort_9b, px.index)
print("\n=== annual_xsec_mom_12_top3 ===")
r = evaluate("annual_xsec_mom_12_top3", w_9b); print_eval(r, SPY_CAGR); results["annual_xsec_mom_12_top3"] = r

In [ ]:
# === 9c: Annual rolling TS momentum (Faber-style, per-country on/off) ===
picks_9c = ts_momentum_12mo(px)  # 1 if 12mo return > 0, else 0
# cohort: equal weight across all 'on' countries, BIL fills if none
cohort_9c = pd.DataFrame(0.0, index=picks_9c.index, columns=COUNTRIES + ["BIL"])
for i, dt in enumerate(picks_9c.index):
    row = picks_9c.iloc[i]
    n_on = int(row.sum())
    if n_on == 0:
        cohort_9c.loc[dt, "BIL"] = 1.0
    else:
        weight = 1.0 / n_on
        for c in picks_9c.columns:
            if row[c] == 1:
                cohort_9c.loc[dt, c] = weight

w_9c = overlapping_12mo_weights(cohort_9c, px.index)
print("=== annual_ts_mom_12mo ===")
r = evaluate("annual_ts_mom_12mo", w_9c); print_eval(r, SPY_CAGR); results["annual_ts_mom_12mo"] = r

In [ ]:
# === 9d: Annual rolling dual momentum, top 1 ===
cohort_9d_raw = dual_momentum(px, top_k=1)  # already in cohort-weight form
# cohort_9d_raw cols are COUNTRIES + ['BIL'] — use it directly
w_9d = overlapping_12mo_weights(cohort_9d_raw, px.index)
print("=== annual_dual_mom_top1 ===")
r = evaluate("annual_dual_mom_top1", w_9d); print_eval(r, SPY_CAGR); results["annual_dual_mom_top1"] = r

### Contrarian (mean reversion) variants — long bottom-N

**⚠️ Underpowered:**
- 3yr contrarian: first valid trade Jan 2019 → ~6.3 years OOS
- 5yr contrarian: first valid trade Jan 2021 → ~4.3 years OOS

In [ ]:
def xsec_contrarian(prices, n_months, bottom_k):
    """Long bottom-K countries by N-month return (mean reversion)."""
    mom = simple_momentum(prices[COUNTRIES], n_months)
    avail = available_mask(prices[COUNTRIES], n_months_required=n_months)
    mom_masked = mom.where(avail)
    
    picks = pd.DataFrame(0, index=mom_masked.index, columns=mom_masked.columns)
    for i, dt in enumerate(mom_masked.index):
        row = mom_masked.iloc[i].dropna()
        if len(row) < bottom_k:
            continue
        bot = row.nsmallest(bottom_k).index
        picks.loc[dt, bot] = 1
    return picks

# === 9e: 3-year contrarian, long bottom 5 ===
picks_9e = xsec_contrarian(px, n_months=36, bottom_k=5)
cohort_9e = picks_to_cohort_weights(picks_9e, COUNTRIES)
w_9e = overlapping_12mo_weights(cohort_9e, px.index)
print("=== annual_contrarian_3yr_bot5 [⚠️ ~6yr OOS] ===")
r = evaluate("annual_contrarian_3yr_bot5", w_9e); print_eval(r, SPY_CAGR); results["annual_contrarian_3yr_bot5"] = r

# === 9f: 5-year contrarian, long bottom 5 ===
picks_9f = xsec_contrarian(px, n_months=60, bottom_k=5)
cohort_9f = picks_to_cohort_weights(picks_9f, COUNTRIES)
w_9f = overlapping_12mo_weights(cohort_9f, px.index)
print("\n=== annual_contrarian_5yr_bot5 [⚠️ ~4yr OOS] ===")
r = evaluate("annual_contrarian_5yr_bot5", w_9f); print_eval(r, SPY_CAGR); results["annual_contrarian_5yr_bot5"] = r

## Section 10 — Rerun master comparison with all strategies

In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        "strategy": name,
        "delta_cagr": r["filled"]["cagr"] - SPY_CAGR,
        "cagr": r["filled"]["cagr"],
        "sharpe": r["filled"]["sharpe"],
        "maxdd": r["filled"]["max_dd"],
        "calmar": r["filled"]["calmar"],
        "p_value": r["p_filled"],
        "pct_active": r["pct_active"],
    })
summary = pd.DataFrame(rows).set_index("strategy").sort_values("delta_cagr", ascending=False)
summary.loc["SPY_BH"] = {"delta_cagr": 0, "cagr": SPY_CAGR, "sharpe": spy_metrics["sharpe"],
                          "maxdd": spy_metrics["max_dd"], "calmar": spy_metrics["calmar"],
                          "p_value": np.nan, "pct_active": 100}

disp = summary.copy()
disp["delta_cagr"] = disp["delta_cagr"].map(lambda x: f"{x:+.1%}" if pd.notna(x) else "")
for c in ["cagr", "maxdd"]:
    disp[c] = disp[c].map(lambda x: f"{x:.1%}" if pd.notna(x) else "")
for c in ["sharpe", "calmar"]:
    disp[c] = disp[c].map(lambda x: f"{x:.2f}" if pd.notna(x) else "")
disp["p_value"] = disp["p_value"].map(lambda x: f"{x:.3f}" if pd.notna(x) else "")
disp["pct_active"] = disp["pct_active"].map(lambda x: f"{x:.0f}%")

print("=" * 120)
print("COUNTRY ROTATION FULL SUMMARY (monthly + annual + contrarian)")
print("=" * 120)
print(disp.to_string())

In [ ]:
# Correlation matrix on filled returns
ret_df = pd.DataFrame({name: r["filled_ret"] for name, r in results.items()})
ret_df["SPY_BH"] = spy_ret_full
corr = ret_df.corr()

fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=8)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=6,
                color="white" if abs(corr.iloc[i,j]) > 0.5 else "black")
plt.colorbar(im, ax=ax, fraction=0.04)
ax.set_title("Daily-return correlation matrix — all country strategies + SPY")
plt.tight_layout(); plt.show()

# Print correlations with SPY specifically
spy_corrs = corr["SPY_BH"].drop("SPY_BH").sort_values()
print("\nCorrelations with SPY (sorted ascending — lowest = best diversifier):")
for name, c in spy_corrs.items():
    print(f"  {name:35s}  {c:+.3f}")

In [ ]:
# Equity curves overlay (focus on annual/contrarian vs originals)
fig, ax = plt.subplots(figsize=(14, 7))
annual_strats = [k for k in results.keys() if k.startswith("annual_")]
for name in annual_strats:
    eq = results[name]["eq_filled"]
    ax.plot(np.asarray(eq.index), np.asarray(eq.values), lw=1.5, alpha=0.8, label=name)
ax.plot(np.asarray(spy_eq_full.index), np.asarray(spy_eq_full.values), lw=2.5, color="black", label="SPY B&H")
ax.set_yscale("log")
ax.set_title("Annual rebal & contrarian strategies vs SPY")
ax.legend(fontsize=9, loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Save updated outputs
import json
summary.to_csv(OUT_DIR / "country_rotation_summary_v2.csv")
out = {}
for name, r in results.items():
    out[name] = {
        "active_days": r["active_days"],
        "pct_active": r["pct_active"],
        "cagr": float(r["filled"]["cagr"]),
        "sharpe": float(r["filled"]["sharpe"]),
        "max_dd": float(r["filled"]["max_dd"]),
        "p_value": float(r["p_filled"]) if pd.notna(r["p_filled"]) else None,
    }
with open(OUT_DIR / "country_rotation_summary_v2.json", "w") as f:
    json.dump(out, f, indent=2)
print(f"Saved → {OUT_DIR}/country_rotation_summary_v2.{{csv,json}}")